In [ ]:
import numpy as np
import math
import time

In [ ]:
kolumny_duże = 3
wiersze_duże = 3
kolumny_małe = 3
wiersze_małe = 3
koniec = None
remis = False
pozycja = False
znak_gracz = str()
tura = str()
następna_plansza = None # współrzędne (k, w) małej planszy, na której trzeba grać

def tworzenie_planszy():
  # 9 plansz 3x3 # plansze[wiersze_duże][kolumny_duże][wiersze_małe][kolumny_małe]
  plansza = [[[[" " for _ in range(3)] for _ in range(3)] for _ in range(3)] for _ in range(3)]
  # 3x3 do śledzenia wygranych w małych kwadratach
  duża_plansza = [[" " for _ in range(3)] for _ in range(3)]

  return plansza, duża_plansza

def wyświetlenie_planszy(plansza):
  for w_duży in range(3):
    print("-" * 25)
    for w_mały in range(3):
      line = ""
      for k_duży in range(3):
        line += "| " + " ".join(plansza[w_duży][k_duży][w_mały]) + " "
      print(line + "|")
  print("-" * 25)


In [ ]:
def sprawdź_wygrana_3x3(sektor):
  for wiersz in range(3):
    if sektor[wiersz][0] == sektor[wiersz][1] == sektor[wiersz][2] != " ":
      return sektor[wiersz][0] # zwraca "X" lub "O"

  for kolumna in range(3):
    if sektor[0][kolumna] == sektor[1][kolumna] == sektor[2][kolumna] != " ":
      return sektor[0][kolumna]

  if sektor[0][0] == sektor[1][1] == sektor[2][2] != " ":
    return sektor[0][0]
  if sektor[0][2] == sektor[1][1] == sektor[2][0] != " ":
    return sektor[0][2]

  return None # nikt nie wygrał

def czy_remis(sektor):
  if sprawdź_wygrana_3x3(sektor) is not None:
    return False
  return all(pole != " " for wiersz in sektor for pole in wiersz)

In [ ]:
def stan_sektora(plansza, duża_plansza, pozycja_d_w, pozycja_d_k):
  mała_p = plansza[pozycja_d_w][pozycja_d_k]
  zwycięzca = sprawdź_wygrana_3x3(mała_p)

  if zwycięzca != None:
    duża_plansza[pozycja_d_w][pozycja_d_k] = zwycięzca
    return True # sektor wypełniony

  if czy_remis(mała_p) == True:
    duża_plansza[pozycja_d_w][pozycja_d_k] = "R"
    return True

  return False # na sektorze można grać

In [ ]:
def wypełnienie(plansza, duża_plansza):
  for d_w in range(3):
    for d_k in range(3):
      stan = duża_plansza[d_w][d_k]
      if stan != " " and stan != "R": # jeśli na małej planszy wygrał O / X
        for m_w in range(3):
          for m_k in range(3):
            plansza[d_w][d_k][m_w][m_k] = stan
      elif stan == "R":
        for m_w in range(3):
          for m_k in range(3):
            plansza[d_w][d_k][m_w][m_k] = "-"

MINMAX

In [ ]:
def heurystyka(plansza, duża_p):
  wynik = 0

  # 1. Punkty za wygranie małej planszy
  for d_w in range(3):
    for d_k in range(3):
      if duża_p[d_w][d_k] == znak_komputer: wynik += 100
      elif duża_p[d_w][d_k] == znak_gracz: wynik -= 100

  linie = [[(0,0), (0,1), (0,2)], [(1,0), (1,1), (1,2)], [(2,0), (2,1), (2,2)], # wiersze
      [(0,0), (1,0), (2,0)], [(0,1), (1,1), (2,1)], [(0,2), (1,2), (2,2)], # kolumny
      [(0,0), (1,1), (2,2)], [(0,2), (1,1), (2,0)]] # przekątne

  for linia in linie:
    symbole = [duża_p[w][k] for w, k in linia]

    # pomijanie linii, gdzie jest remis, bo tam już i tak nikt nie wygra
    if "R" in symbole:
      continue

    # 2. Punkty za wygranie dwóch małych plansz w linii
    # dwa pola komputera i jeden pusty (bez blokady przez gracza)
    if symbole.count(znak_komputer) == 2 and symbole.count(" ") == 1: wynik += 200
    if symbole.count(znak_gracz) == 2 and symbole.count(" ") == 1: wynik -= 200

    # 3. Punkty za blokowanie wygranej przeciwnika
    if symbole.count(znak_gracz) == 2 and symbole.count(znak_komputer) == 1: wynik += 150
    if symbole.count(znak_komputer) == 2 and symbole.count(znak_gracz) == 1: wynik -= 150

  for d_w in range(3):
    for d_k in range(3):
      if duża_p[d_w][d_k] == " ":
        mała_plansza = plansza[d_w][d_k]
        for linia in linie:
          symbole_małe = [mała_plansza[m_w][m_k] for m_w, m_k in linia]
          # 4. Punkty za dwa znaki w linii na małej planszy
          if symbole_małe.count(znak_komputer) == 2 and symbole_małe.count(" ") == 1: wynik += 5
          if symbole_małe.count(znak_gracz) == 2 and symbole_małe.count(" ") == 1: wynik -= 5

          # 5. Punkty za blokowanie wygranej przeciwnika na małej planszy
          if symbole_małe.count(znak_gracz) == 2 and symbole_małe.count(znak_komputer) == 1: wynik += 20
          if symbole_małe.count(znak_komputer) == 2 and symbole_małe.count(znak_gracz) == 1: wynik -= 20

  return wynik

In [ ]:
def minimax_alpha_beta(plansza, duża_p, następna_p, depth, alpha, beta, maksymalizacja):
  # sprawdzanie, czy gra się skończyła lub czy osiągnęto limit głębokości
  wynik = sprawdź_wygrana_3x3(duża_p)
  if wynik == znak_komputer: return 1000 + depth, None # wygrana komputera
  if wynik == znak_gracz: return -1000 - depth, None   # wygrana gracza
  if depth == 0 or czy_remis(duża_p):
    return heurystyka(plansza, duża_p), None

  # ustalanie małych plansz, w których można grać
  if następna_p is None or duża_p[następna_p[0]][następna_p[1]] != " ":
    możliwe_sektory = [(w, k) for w in range(3) for k in range(3) if duża_p[w][k] == " "]
  else:
    możliwe_sektory = [następna_p]

  if maksymalizacja == True:
    najlepsza_wartość = -math.inf
    najlepszy_ruch = None

    for (d_w, d_k) in możliwe_sektory:
      for m_w in range(3):
        for m_k in range(3):
          if plansza[d_w][d_k][m_w][m_k] == " ":
            stary_stan = duża_p[d_w][d_k]
            plansza[d_w][d_k][m_w][m_k] = znak_komputer
            wynik_tego_sektora = sprawdź_wygrana_3x3(plansza[d_w][d_k])
            if wynik_tego_sektora is None and czy_remis(plansza[d_w][d_k]) == True:
              wynik_tego_sektora = "R"
            if wynik_tego_sektora is not None:
              duża_p[d_w][d_k] = wynik_tego_sektora

            wynik_ruchu = minimax_alpha_beta(plansza, duża_p, (m_w, m_k), depth - 1, alpha, beta, False)[0]

            # cofnieęcie wartości pól
            plansza[d_w][d_k][m_w][m_k] = " "
            duża_p[d_w][d_k] = stary_stan

            if wynik_ruchu > najlepsza_wartość:
              najlepsza_wartość = wynik_ruchu
              najlepszy_ruch = (d_w, d_k, m_w, m_k)

            alpha = max(alpha, najlepsza_wartość)
            if beta <= alpha:
              break # odcięcie Alpha
        if beta <= alpha: break
      if beta <= alpha: break
    return najlepsza_wartość, najlepszy_ruch

  else:
    najlepsza_wartość = math.inf
    najlepszy_ruch = None

    for (d_w, d_k) in możliwe_sektory:
      for m_w in range(3):
        for m_k in range(3):
          if plansza[d_w][d_k][m_w][m_k] == " ":
            stary_stan = duża_p[d_w][d_k]
            plansza[d_w][d_k][m_w][m_k] = znak_gracz
            wynik_tego_sektora = sprawdź_wygrana_3x3(plansza[d_w][d_k])
            if wynik_tego_sektora is None and czy_remis(plansza[d_w][d_k]):
              wynik_tego_sektora = "R"
            if wynik_tego_sektora is not None:
              duża_p[d_w][d_k] = wynik_tego_sektora

            wynik_ruchu = minimax_alpha_beta(plansza, duża_p, (m_w, m_k), depth - 1, alpha, beta, True)[0]

            plansza[d_w][d_k][m_w][m_k] = " "
            duża_p[d_w][d_k] = stary_stan

            if wynik_ruchu < najlepsza_wartość:
              najlepsza_wartość = wynik_ruchu
              najlepszy_ruch = (d_w, d_k, m_w, m_k)

            beta = min(beta, najlepsza_wartość)
            if beta <= alpha:
              break
        if beta <= alpha: break
      if beta <= alpha: break
    return najlepsza_wartość, najlepszy_ruch

GRA

In [ ]:
plansza, duża_plansza = tworzenie_planszy()

while znak_gracz != "X" and znak_gracz != "O":
  znak_gracz = input("Wybierz X lub O: ").upper()

znak_komputer = "O" if znak_gracz == "X" else "X"

while tura != "T" and tura != "N":
  tura = input("Czy chcesz zaczynać? (T/N): ").upper()

tura = "gracz" if tura == "T" else "komputer"

GRACZ VS KOMPUTER

In [ ]:
wyświetlenie_planszy(plansza)

while koniec == None and remis == False:

  if tura == "gracz":
    print(f"\nRuch gracza: {znak_gracz}.")
    # wybór dużej planszy, jeżeli początek gry lub dana mała plansza jest już wygrana
    if następna_plansza is None or duża_plansza[następna_plansza[0]][następna_plansza[1]] != " ":
      print("Możesz wybrać dowolną małą planszę!")
      while pozycja != True:
        pozycja_d_w = int(input("Wybierz duży wiersz (1-3): ")) - 1
        pozycja_d_k = int(input("Wybierz dużą kolumnę (1-3): ")) - 1
        if duża_plansza[pozycja_d_w][pozycja_d_k] == " ":
          pozycja = True
        else:
          print(f"Sektor [{pozycja_d_w+1}, {pozycja_d_k+1}] jest już zajęty! Wybierz inny.")
    else:
        pozycja_d_w, pozycja_d_k = następna_plansza
        print(f"Musisz grać na planszy: [{pozycja_d_w+1}, {pozycja_d_k+1}]")

    pozycja = False
    while pozycja != True:
      pozycja_m_w = int(input("Podaj wiersz (1-3):"))-1
      pozycja_m_k = int(input("Podaj kolumnę (1-3):"))-1
      if plansza[pozycja_d_w][pozycja_d_k][pozycja_m_w][pozycja_m_k] == " ":
        pozycja = True
      else:
        print("To pole jest już zajęte! Wybierz inne.")

    następna_plansza = (pozycja_m_w, pozycja_m_k)
    plansza[pozycja_d_w][pozycja_d_k][pozycja_m_w][pozycja_m_k] = znak_gracz
    stan_sektora(plansza, duża_plansza, pozycja_d_w, pozycja_d_k)
    wypełnienie(plansza, duża_plansza)
    wyświetlenie_planszy(plansza)

    pozycja = False

    koniec = sprawdź_wygrana_3x3(duża_plansza)
    remis = czy_remis(duża_plansza)

  else:
    print(f"\nRuch komputera: {znak_komputer}.")
    ocena, ruch = minimax_alpha_beta(plansza, duża_plansza, następna_plansza, 6, -math.inf, math.inf, False)

    if ruch != None:
      pozycja_d_w, pozycja_d_k, pozycja_m_w, pozycja_m_k = ruch
      plansza[pozycja_d_w][pozycja_d_k][pozycja_m_w][pozycja_m_k] = znak_komputer
      następna_plansza = (pozycja_m_w, pozycja_m_k)
      stan_sektora(plansza, duża_plansza, pozycja_d_w, pozycja_d_k)
      wypełnienie(plansza, duża_plansza)

      print(f"Komputer zagrał na duża plansza: [{pozycja_d_w+1},{pozycja_d_k+1}], mała plansza:[{pozycja_m_w+1},{pozycja_m_k+1}]")
      wyświetlenie_planszy(plansza)
      time.sleep(0.5)
    else:
      print("Komputer nie znalazł ruchu!")

    koniec = sprawdź_wygrana_3x3(duża_plansza)
    remis = czy_remis(duża_plansza)

  tura = "komputer" if tura == "gracz" else "gracz"

if remis == True:
  print("Remis! Gra nierozstrzygnięta.")
else:
  if koniec == znak_komputer:
    print(f"Wygrał komputer: {znak_komputer}!")
  else:
    print(f"Wygrał gracz: {znak_gracz}!")